<a href="https://colab.research.google.com/github/gawlowskiandrzej/Eksploracja_danych_projekt---detekcja-spamu/blob/main/spam_dataset_quality_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spam Dataset Quality Analysis Notebook

This notebook evaluates dataset quality using:
- Lexical overlap (Jaccard similarity)
- Feature leakage (Chi-square)
- Entropy analysis
- Length analysis
- Duplicate detection
- Baseline model sanity check


In [21]:
import os
from collections import Counter

import kagglehub
import numpy as np
import pandas as pd

from kagglehub import KaggleDatasetAdapter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_score


In [22]:
DATA_DIR = "data"
TEXT_COL = "text"
LABEL_COL = "label"
IS_SPAM_COL = "is_spam"

DEFAULT_SPAM_VALUES = [1, "1", True, "true", "yes", "spam", "phishing", "scam"]
DEFAULT_HAM_VALUES = [0, "0", False, "false", "no", "ham", "not spam", "not_spam", "legit", "legitimate"]

DATASET_CONFIGS = [
    {
        "name": "Kaggle - Spam Email Detection",
        "source": "kaggle",
        "kaggle_dataset": "ssssws/spam-email-detection-dataset-clean-and-ml-ready",
        "file_name": "spam_email_dataset.csv",
        "cache_file": "spam_email_dataset.csv",
        "text_col": "email_text",
        "label_col": "label",
        "spam_values": [1],
        "ham_values": [0],
    },
    {
        "name": "Local Spam Email Dataset",
        "source": "local",
        "file_name": "email_dataset_100k.csv",
        "text_col": "body_plain",
        "label_col": "label",
        "spam_values": [1],
        "ham_values": [0],
    },
    {
        "name": "Local (LLM)",
        "source": "local",
        "file_name": "llm_spam_email_dataset.csv",
        "text_col": "email_text",
        "label_col": "label",
        "spam_values": [1],
        "ham_values": [0],
    }
    # Example for another Kaggle dataset:
    # {
    #     "name": "Kaggle - Example Spam/Ham",
    #     "source": "kaggle",
    #     "kaggle_dataset": "owner/dataset-slug",
    #     "file_name": "emails.csv",
    #     "cache_file": "example_spam_ham.csv",
    #     "text_col": "message",
    #     "label_col": "category",
    #     "spam_values": ["spam"],
    #     "ham_values": ["ham"],
    # },
]

os.makedirs(DATA_DIR, exist_ok=True)

def normalize_label_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value)).lower()
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    return str(value).strip().lower()

def normalized_value_set(values):
    return {normalize_label_value(value) for value in values}

def map_labels_to_binary(label_series, spam_values=None, ham_values=None):
    spam_set = normalized_value_set(spam_values or DEFAULT_SPAM_VALUES)
    ham_set = normalized_value_set(ham_values or DEFAULT_HAM_VALUES)
    unknown_values = set()

    def map_one(value):
        normalized = normalize_label_value(value)
        if normalized in spam_set:
            return 1
        if normalized in ham_set:
            return 0
        unknown_values.add(str(value))
        return np.nan

    mapped = label_series.apply(map_one)
    if unknown_values:
        preview = sorted(unknown_values)[:10]
        raise ValueError(
            "Unknown label values: "
            f"{preview}. Add them to spam_values or ham_values in DATASET_CONFIGS."
        )
    return mapped.astype(int)

def cache_file_for(config):
    return config.get("cache_file") or config["file_name"].replace("/", "_").replace("\\", "_")

def load_raw_dataset(config):
    source = config["source"]
    file_name = config["file_name"]
    cache_path = os.path.join(DATA_DIR, cache_file_for(config))

    if source == "local":
        if not os.path.exists(cache_path):
            raise FileNotFoundError(f"Missing local file: {cache_path}")
        print(f"Loading local dataset: {config['name']}")
        return pd.read_csv(cache_path)

    if source == "kaggle":
        if os.path.exists(cache_path):
            print(f"Loading cached Kaggle dataset: {cache_path}")
            return pd.read_csv(cache_path)

        print(f"Downloading Kaggle dataset: {config['name']}...")
        df = kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            config["kaggle_dataset"],
            file_name,
        )
        df.to_csv(cache_path, index=False)
        return df

    raise ValueError(f"Unsupported dataset source: {source}")

def standardize_dataset(df, config):
    text_col = config["text_col"]
    label_col = config["label_col"]
    missing_cols = [col for col in [text_col, label_col] if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Dataset {config['name']} is missing columns: {missing_cols}")

    mapped_labels = map_labels_to_binary(
        df[label_col],
        spam_values=config.get("spam_values"),
        ham_values=config.get("ham_values"),
    )

    standardized = pd.DataFrame({
        "dataset": config["name"],
        TEXT_COL: df[text_col].fillna("").astype(str),
        LABEL_COL: mapped_labels,
        IS_SPAM_COL: mapped_labels.astype(bool),
    })
    return standardized[standardized[TEXT_COL].str.strip().ne("")].reset_index(drop=True)

loaded_datasets = []
for config in DATASET_CONFIGS:
    raw_df = load_raw_dataset(config)
    df = standardize_dataset(raw_df, config)
    loaded_datasets.append({
        "name": config["name"],
        "config": config,
        "df": df,
        "raw_rows": len(raw_df),
        "rows": len(df),
    })

datasets = {dataset["name"]: dataset["df"] for dataset in loaded_datasets}

dataset_overview = pd.DataFrame([
    {
        "Dataset": dataset["name"],
        "Source": dataset["config"]["source"],
        "Text column": dataset["config"]["text_col"],
        "Label column": dataset["config"]["label_col"],
        "Raw rows": dataset["raw_rows"],
        "Rows used": dataset["rows"],
        "Spam rows": int((dataset["df"][LABEL_COL] == 1).sum()),
        "Ham rows": int((dataset["df"][LABEL_COL] == 0).sum()),
    }
    for dataset in loaded_datasets
])

dataset_overview


Loading cached Kaggle dataset: data\spam_email_dataset.csv
Loading local dataset: Local Spam Email Dataset
Loading local dataset: Local (LLM)


,Dataset,Source,Text column,Label column,Raw rows,Rows used,Spam rows,Ham rows
0,Kaggle - Spam Email Detection,kaggle,email_text,label,10000,10000,3995,6005
1,Local Spam Email Dataset,local,body_plain,label,100000,100000,50000,50000
2,Local (LLM),local,email_text,label,100,100,48,52


In [23]:
def compute_jaccard(df):
    spam_words = set(" ".join(df[df[LABEL_COL] == 1][TEXT_COL]).lower().split())
    ham_words = set(" ".join(df[df[LABEL_COL] == 0][TEXT_COL]).lower().split())
    all_words = spam_words | ham_words
    jaccard = len(spam_words & ham_words) / len(all_words) if all_words else np.nan

    if pd.isna(jaccard):
        conclusion = "not enough vocabulary"
    elif jaccard < 0.2:
        conclusion = "🔴 very low overlap (easy / keyword-driven)"
    elif jaccard < 0.5:
        conclusion = "🟡 moderate overlap (realistic)"
    else:
        conclusion = "🟢 high overlap (hard / semantic)"

    return jaccard, conclusion, len(spam_words), len(ham_words)

rows = []
for dataset in loaded_datasets:
    jaccard, conclusion, spam_vocab, ham_vocab = compute_jaccard(dataset["df"])
    rows.append({
        "Dataset": dataset["name"],
        "Jaccard Similarity": jaccard,
        "Spam vocab size": spam_vocab,
        "Ham vocab size": ham_vocab,
        "Conclusion": conclusion,
    })

comparison_df = pd.DataFrame(rows)
comparison_df


,Dataset,Jaccard Similarity,Spam vocab size,Ham vocab size,Conclusion
0,Kaggle - Spam Email Detection,0.988730,1935,1947,🟢 high overlap (hard / semantic)
1,Local Spam Email Dataset,0.000443,95014,20048,🔴 very low overlap (easy / keyword-driven)
2,Local (LLM),0.210151,743,783,🟡 moderate overlap (realistic)


In [24]:
KEYWORDS = ["free", "win", "click", "urgent", "offer"]

def chi2_analysis(df):
    if df[LABEL_COL].nunique() < 2:
        return [], 0, "not enough classes"

    try:
        cv = CountVectorizer(stop_words="english", min_df=5)
        X_counts = cv.fit_transform(df[TEXT_COL])
        chi_scores, _ = chi2(X_counts, df[LABEL_COL])
    except ValueError as exc:
        return [], 0, f"skipped: {exc}"

    features_all = np.array(cv.get_feature_names_out())
    top_idx = chi_scores.argsort()[-20:][::-1]
    features = features_all[top_idx].tolist()
    keyword_hits = sum(feature in KEYWORDS for feature in features)

    if keyword_hits >= 10:
        conclusion = "🔴 strong keyword reliance (too easy)"
    elif keyword_hits >= 5:
        conclusion = "🟡 partial keyword patterns"
    else:
        conclusion = "🟢 low keyword leakage (semantic)"

    return features, keyword_hits, conclusion

rows = []
for dataset in loaded_datasets:
    features, keyword_hits, conclusion = chi2_analysis(dataset["df"])
    rows.append({
        "Dataset": dataset["name"],
        "Keyword hits (top20)": keyword_hits,
        "Conclusion": conclusion,
        "Top features": ", ".join(features),
    })

comparison_df = pd.DataFrame(rows)
comparison_df


,Dataset,Keyword hits (top20),Conclusion,Top features
0,Kaggle - Spam Email Detection,5,🟡 partial keyword patterns,"urgent, guarantee, limited, win, click, cash, ..."
1,Local Spam Email Dataset,1,🟢 low keyword leakage (semantic),"https, ref, thanks, verify, account, attached,..."
2,Local (LLM),1,🟢 low keyword leakage (semantic),"click, information, link, verify, activity, ho..."


In [25]:
def entropy(texts):
    words = " ".join(texts.astype(str)).lower().split()
    if not words:
        return np.nan
    freq = Counter(words)
    probs = np.array(list(freq.values())) / sum(freq.values())
    return -np.sum(probs * np.log2(probs))

def entropy_analysis(df):
    spam_ent = entropy(df[df[LABEL_COL] == 1][TEXT_COL])
    ham_ent = entropy(df[df[LABEL_COL] == 0][TEXT_COL])
    diff = abs(spam_ent - ham_ent)

    if pd.isna(diff):
        conclusion = "🔴not enough text in both classes"
    elif diff < 0.2:
        conclusion = "🟢 balanced linguistic complexity"
    elif spam_ent > ham_ent:
        conclusion = "🟡 spam more diverse"
    else:
        conclusion = "🔴 entropy imbalance (bias/artifacts)"

    return spam_ent, ham_ent, diff, conclusion

rows = []
for dataset in loaded_datasets:
    spam_ent, ham_ent, diff, conclusion = entropy_analysis(dataset["df"])
    rows.append({
        "Dataset": dataset["name"],
        "Spam entropy": spam_ent,
        "Ham entropy": ham_ent,
        "Abs diff": diff,
        "Conclusion": conclusion,
    })

comparison_df = pd.DataFrame(rows)
comparison_df


,Dataset,Spam entropy,Ham entropy,Abs diff,Conclusion
0,Kaggle - Spam Email Detection,7.456187,10.007867,2.551680,🔴 entropy imbalance (bias/artifacts)
1,Local Spam Email Dataset,8.625144,7.606818,1.018326,🟡 spam more diverse
2,Local (LLM),8.376785,8.489771,0.112986,🟢 balanced linguistic complexity


In [26]:
def length_analysis(df):
    df = df.copy()
    df["length"] = df[TEXT_COL].apply(len)

    spam_mean = df[df[LABEL_COL] == 1]["length"].mean()
    ham_mean = df[df[LABEL_COL] == 0]["length"].mean()
    ratio = spam_mean / (ham_mean + 1e-6) if pd.notna(spam_mean) and pd.notna(ham_mean) else np.nan

    if pd.isna(ratio):
        conclusion = "🔴 not enough classes"
    elif 0.8 <= ratio <= 1.2:
        conclusion = "🟢 similar lengths (realistic)"
    elif ratio > 1.5:
        conclusion = "🟡 spam much longer (bias)"
    else:
        conclusion = "🔴 strong length imbalance"

    return spam_mean, ham_mean, ratio, conclusion

rows = []
for dataset in loaded_datasets:
    spam_mean, ham_mean, ratio, conclusion = length_analysis(dataset["df"])
    rows.append({
        "Dataset": dataset["name"],
        "Spam mean length": spam_mean,
        "Ham mean length": ham_mean,
        "Spam/Ham ratio": ratio,
        "Conclusion": conclusion,
    })

comparison_df = pd.DataFrame(rows)
comparison_df


,Dataset,Spam mean length,Ham mean length,Spam/Ham ratio,Conclusion
0,Kaggle - Spam Email Detection,131.297622,104.278102,1.259110,🔴 strong length imbalance
1,Local Spam Email Dataset,144.800680,102.173300,1.417207,🔴 strong length imbalance
2,Local (LLM),226.479167,202.538462,1.118203,🟢 similar lengths (realistic)


In [27]:
def similarity_analysis(df, sample_size=2000):
    df = df.copy()
    if len(df) > sample_size:
        df = df.sample(sample_size, random_state=42)

    if len(df) < 2:
        return np.nan, "not enough rows", len(df)

    try:
        tfidf = TfidfVectorizer().fit_transform(df[TEXT_COL])
    except ValueError as exc:
        return np.nan, f"skipped: {exc}", len(df)

    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0)
    max_sim = sim.max()

    if max_sim > 0.9:
        conclusion = "🔴 very high duplication (low quality / synthetic)"
    elif max_sim > 0.75:
        conclusion = "🟡 moderate duplication"
    else:
        conclusion = "🟢 low duplication (good diversity)"

    return max_sim, conclusion, len(df)

rows = []
for dataset in loaded_datasets:
    max_sim, conclusion, sample_n = similarity_analysis(dataset["df"])
    rows.append({
        "Dataset": dataset["name"],
        "Max cosine similarity": max_sim,
        "Sample size": sample_n,
        "Conclusion": conclusion,
    })

comparison_df = pd.DataFrame(rows)
comparison_df


,Dataset,Max cosine similarity,Sample size,Conclusion
0,Kaggle - Spam Email Detection,0.591229,2000,🟢 low duplication (good diversity)
1,Local Spam Email Dataset,1.000000,2000,🔴 very high duplication (low quality / synthetic)
2,Local (LLM),0.653389,100,🟢 low duplication (good diversity)


In [28]:
def cv_analysis(df):
    class_counts = df[LABEL_COL].value_counts()
    if len(class_counts) < 2:
        return np.nan, "not enough classes"

    cv_folds = min(5, int(class_counts.min()))
    if cv_folds < 2:
        return np.nan, "not enough rows per class"

    try:
        vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
        X = vectorizer.fit_transform(df[TEXT_COL])
        model = LogisticRegression(max_iter=1000)
        scores = cross_val_score(model, X, df[LABEL_COL], cv=cv_folds)
    except ValueError as exc:
        return np.nan, f"skipped: {exc}"

    acc = scores.mean()
    if acc > 0.95:
        conclusion = "🔴 likely leakage / too easy dataset"
    elif acc > 0.85:
        conclusion = "🟡 learnable patterns but some structure"
    else:
        conclusion = "🟢 harder / more semantic reasoning required"

    return acc, conclusion

rows = []
for dataset in loaded_datasets:
    accuracy, conclusion = cv_analysis(dataset["df"])
    rows.append({
        "Dataset": dataset["name"],
        "CV Accuracy": accuracy,
        "Conclusion": conclusion,
    })

comparison_df = pd.DataFrame(rows)
comparison_df


,Dataset,CV Accuracy,Conclusion
0,Kaggle - Spam Email Detection,1.00,🔴 likely leakage / too easy dataset
1,Local Spam Email Dataset,1.00,🔴 likely leakage / too easy dataset
2,Local (LLM),0.61,🟢 harder / more semantic reasoning required
